In [1]:
import csv
import pandas as pd

BASE_PATH = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\data\msmarco_passage"

COLLECTION_PATH = BASE_PATH + r"\collection.tsv"
QRELS_PATH = BASE_PATH + r"\qrels.train.tsv"
QUERIES_PATH = BASE_PATH + r"\queries.train.tsv"

PASSAGE_OUT = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\subset_passages.tsv"
QUERY_OUT = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\subset_queries.tsv"

MAX_QUERIES = 30000


print("Loading qrels (correct 4-column format)...")
qrels = pd.read_csv(
    QRELS_PATH,
    sep="\t",
    names=["query_id", "unused", "passage_id", "relevance"],
    dtype=str
)

# Keep only relevant
qrels = qrels[qrels["relevance"] == "1"]

print(f"Total relevant (query, passage) pairs: {len(qrels)}")

# Collect unique query IDs
valid_query_ids = qrels["query_id"].unique()

print(f"Total unique queries with relevance: {len(valid_query_ids)}")

# Limit queries
valid_query_ids = set(valid_query_ids[:MAX_QUERIES])

print(f"Using query_ids: {len(valid_query_ids)}")


print("Extracting subset queries...")
subset_queries = []

with open(QUERIES_PATH, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for qid, query in reader:
        if qid in valid_query_ids:
            subset_queries.append((qid, query))
        if len(subset_queries) >= MAX_QUERIES:
            break

pd.DataFrame(
    subset_queries,
    columns=["query_id", "query"]
).to_csv(QUERY_OUT, sep="\t", index=False)

print(f"Saved subset queries: {len(subset_queries)} to {QUERY_OUT}")


print("Extracting subset passages...")
passage_id_set = set(
    qrels[qrels["query_id"].isin(valid_query_ids)]["passage_id"]
)

subset_passages = []

with open(COLLECTION_PATH, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for pid, passage in reader:
        if pid in passage_id_set:
            subset_passages.append((pid, passage))
        if len(subset_passages) >= len(passage_id_set):
            break

pd.DataFrame(
    subset_passages,
    columns=["passage_id", "passage"]
).to_csv(PASSAGE_OUT, sep="\t", index=False)

print(f"Saved subset passages: {len(subset_passages)} → {PASSAGE_OUT}")


Loading qrels (correct 4-column format)...
Total relevant (query, passage) pairs: 532761
Total unique queries with relevance: 502939
Using query_ids: 30000
Extracting subset queries...
Saved subset queries: 30000 to C:\Users\nihal\OneDrive\Desktop\NLP_Project\subset_queries.tsv
Extracting subset passages...
Saved subset passages: 31953 → C:\Users\nihal\OneDrive\Desktop\NLP_Project\subset_passages.tsv
